In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab data access)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Experimental LightGBM Stacking Pipeline with Optuna Feature Optimization (`models/train_oof_logistic_regression_stacking_exp.ipynb`)

This notebook implements automated **Optuna Feature Selection on the Validation Set** targeting **higher Macro Balanced Accuracy** without SMOTE resampling:

### 🔬 Key Experimental Design
1. **3-Way Stratified Data Structure**: Train (98%), Validation (1%), and Holdout Test (1%) with strict complete-cases filtering.
2. **Expanded Candidate Feature Pool (57 Features Total)**:
   - 15 Core raw vitals & demographics (always active)
   - 23 Baseline engineered threshold flags & range ratios
   - 19 Advanced composite severity features (NEWS score, Euclidean Vital Distance, Multi-system derangement counts, Perfusion gap, Respiratory reserve, Age interactions, etc.)
3. **No Synthetic Over-sampling (SMOTE Removed)**:
   - Sub-models train directly on genuine clinical distributions using native LightGBM optimization, boosting training speed and biological plausibility.
4. **Validation-Driven Optuna Optimization**:
   - Feature combination search evaluated strictly on the **Validation Set (1%)** using 5-Fold Cross-Validation.
   - **Uncapped Execution**: No timeout cutoff (`timeout=None`), runs until all trials complete.
   - Optimization Target: **Macro Balanced Accuracy** across all 5 ESI triage acuity levels.
5. **Production Model Training & Holdout Test Benchmark**:
   - Fits final sub-models and calibrated meta-learner on the winning feature subset.
   - Evaluates on the unseen **Holdout Test Set (1%)** with Confusion Matrix, ROC-AUC, Density Distributions, and Optuna Trajectory.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Raw Dataset, Filter Complete Cases & Stratified 3-Way Split (Train/Val/Test)
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(dplyr)
})

config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) config_path <- "config/triage_conf.json"
config <- fromJSON(config_path)

set.seed(config$training$random_state)

stratified_sample <- function(y, fraction, seed = 42) {
  set.seed(seed)
  idx_list <- split(seq_along(y), y)
  sampled <- unlist(lapply(idx_list, function(idx) {
    n_sample <- max(1, round(length(idx) * fraction))
    sample(idx, size = n_sample)
  }))
  return(sort(sampled))
}

data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) data_file <- paste0("../", data_file)

data_env <- new.env()
load(data_file, envir = data_env)

df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
raw_df   <- get(df_names[which.max(df_sizes)], envir = data_env)
target_col_name <- config$classes$target_col

initial_total_rows <- nrow(raw_df)
cat("========================================================================\n")
cat(sprintf("  INITIAL DATASET LOADED: %d Total Rows, %d Total Columns\n", initial_total_rows, ncol(raw_df)))
cat("========================================================================\n")
if (target_col_name %in% names(raw_df)) {
  cat("Initial ESI Target Distribution (including NAs):\n")
  print(table(raw_df[[target_col_name]], useNA = "ifany"))
  cat("------------------------------------------------------------------------\n")
}

gender_vec <- if ("gender" %in% names(raw_df)) ifelse(is.na(raw_df$gender), NA, ifelse(as.character(raw_df$gender) == "Male", 1, 0)) else rep(NA, nrow(raw_df))
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) raw_df$cc_breathingdifficulty else rep(NA, nrow(raw_df))

get_vec <- function(col_name) {
  if (col_name %in% names(raw_df)) {
    return(raw_df[[col_name]])
  } else {
    return(rep(NA, nrow(raw_df)))
  }
}

raw_esi <- as.character(raw_df[[target_col_name]])

df_master <- data.frame(
  age                     = raw_df$age,
  cc_breathingdifficulty  = cc_bd_vec,
  gender                  = gender_vec,
  triage_vital_hr         = get_vec("triage_vital_hr"),
  triage_vital_sbp        = get_vec("triage_vital_sbp"),
  triage_vital_rr         = get_vec("triage_vital_rr"),
  triage_vital_o2         = get_vec("triage_vital_o2"),
  pulse_min               = get_vec("pulse_min"),
  resp_min                = get_vec("resp_min"),
  spo2_min                = get_vec("spo2_min"),
  sbp_min                 = get_vec("sbp_min"),
  pulse_max               = get_vec("pulse_max"),
  resp_max                = get_vec("resp_max"),
  spo2_max                = get_vec("spo2_max"),
  sbp_max                 = get_vec("sbp_max"),
  target_col              = factor(raw_esi, levels = c("1", "2", "3", "4", "5"))
)

# Strictly drop any row containing at least 1 null/NA value across all 15 raw features or target
df_master <- na.omit(df_master)
df_master$target_num <- as.numeric(as.character(df_master$target_col))

clean_total_rows <- nrow(df_master)
dropped_rows     <- initial_total_rows - clean_total_rows

cat(sprintf("Missing Values Filter: Dropped %d rows with >= 1 NA feature (Retained %d Complete Cases, %.2f%%)\n", 
            dropped_rows, clean_total_rows, (clean_total_rows / initial_total_rows) * 100))
cat("Cleaned ESI Distribution (100% complete cases):\n")
print(table(df_master$target_col))
cat("------------------------------------------------------------------------\n")

# Stratified 3-Way Partitioning based on triage_conf.json
test_size <- config$training$test_size
val_size  <- config$training$val_size
seed_val  <- config$training$random_state

# 1. Extract Stratified Holdout Test Set (e.g., 1%)
idx_test <- stratified_sample(df_master$target_col, test_size, seed = seed_val)
test_df_clean  <- df_master[idx_test, ]
rem_df         <- df_master[-idx_test, ]

# 2. Extract Stratified Validation Set from remainder (e.g., 1% of total)
val_adj_fraction <- val_size / (1 - test_size)
idx_val <- stratified_sample(rem_df$target_col, val_adj_fraction, seed = seed_val + 1)
val_df_clean   <- rem_df[idx_val, ]
train_df_clean <- rem_df[-idx_val, ]

train_mat_export <- as.matrix(cbind(train_df_clean[, 1:15], target = train_df_clean$target_num))
val_mat_export   <- as.matrix(cbind(val_df_clean[, 1:15],   target = val_df_clean$target_num))
test_mat_export  <- as.matrix(cbind(test_df_clean[, 1:15],  target = test_df_clean$target_num))

cat(sprintf("3-Way Partition Complete:\n  Train Set      = %d rows (%.2f%%)\n  Validation Set = %d rows (%.2f%%)\n  Holdout Test   = %d rows (%.2f%%)\n", 
            nrow(train_mat_export), (nrow(train_mat_export) / clean_total_rows) * 100,
            nrow(val_mat_export),   (nrow(val_mat_export) / clean_total_rows) * 100,
            nrow(test_mat_export),  (nrow(test_mat_export) / clean_total_rows) * 100))
cat("========================================================================\n")

In [ ]:
# ---------------------------------------------------------
# Step 2: Build Full Candidate Feature Engineering Pool (57 Features)
# ---------------------------------------------------------
import os
import pickle
import numpy as np
import pandas as pd
from rpy2.robjects import r
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss, balanced_accuracy_score
import lightgbm as lgb
import optuna

train_mat_in = np.array(r('train_mat_export'), dtype=np.float64)
val_mat_in   = np.array(r('val_mat_export'),   dtype=np.float64)
test_mat_in  = np.array(r('test_mat_export'),  dtype=np.float64)

raw_mat_tr  = train_mat_in[:, :15]
y_train     = train_mat_in[:, 15].astype(int)

raw_mat_val = val_mat_in[:, :15]
y_val       = val_mat_in[:, 15].astype(int)

raw_mat_ts  = test_mat_in[:, :15]
y_test      = test_mat_in[:, 15].astype(int)

# NEWS Scoring Helper Functions
def calc_news_hr(hr):
    score = np.zeros_like(hr, dtype=np.float64)
    score[hr <= 40] = 3.0
    score[(hr > 40) & (hr <= 50)] = 1.0
    score[(hr > 90) & (hr <= 110)] = 1.0
    score[(hr > 110) & (hr <= 130)] = 2.0
    score[hr > 130] = 3.0
    return score

def calc_news_rr(rr):
    score = np.zeros_like(rr, dtype=np.float64)
    score[rr <= 8] = 3.0
    score[(rr > 8) & (rr <= 11)] = 1.0
    score[(rr > 20) & (rr <= 24)] = 2.0
    score[rr > 24] = 3.0
    return score

def calc_news_sbp(sbp):
    score = np.zeros_like(sbp, dtype=np.float64)
    score[sbp <= 90] = 3.0
    score[(sbp > 90) & (sbp <= 100)] = 2.0
    score[(sbp > 100) & (sbp <= 110)] = 1.0
    score[sbp > 219] = 3.0
    return score

def calc_news_spo2(o2):
    score = np.zeros_like(o2, dtype=np.float64)
    score[o2 <= 91] = 3.0
    score[(o2 > 91) & (o2 <= 93)] = 2.0
    score[(o2 > 93) & (o2 <= 95)] = 1.0
    return score

def build_candidate_feature_pool(raw_mat):
    N = len(raw_mat)
    TOTAL_CANDIDATES = 57
    X = np.zeros((N, TOTAL_CANDIDATES), dtype=np.float64)
    
    # Core 15 raw features
    X[:, :15] = raw_mat
    age       = raw_mat[:, 0]
    cc_bd     = raw_mat[:, 1]
    gender    = raw_mat[:, 2]
    t_hr      = raw_mat[:, 3]
    t_sbp     = raw_mat[:, 4]
    t_rr      = raw_mat[:, 5]
    t_o2      = raw_mat[:, 6]
    pulse_min = raw_mat[:, 7]
    resp_min  = raw_mat[:, 8]
    spo2_min  = raw_mat[:, 9]
    sbp_min   = raw_mat[:, 10]
    pulse_max = raw_mat[:, 11]
    resp_max  = raw_mat[:, 12]
    spo2_max  = raw_mat[:, 13]
    sbp_max   = raw_mat[:, 14]
    
    hr_rng   = pulse_max - pulse_min
    rr_rng   = resp_max - resp_min
    spo2_rng = spo2_max - spo2_min
    sbp_rng  = sbp_max - sbp_min
    
    # [15..24] Baseline Single-Vital Threshold Flags
    is_dyspnea_total       = (t_o2 < 90).astype(float)
    is_dyspnea_moderate    = ((t_o2 >= 90) & (t_o2 < 94)).astype(float)
    is_bradypnea           = (t_rr < 10).astype(float)
    is_tachypnea           = (t_rr > 30).astype(float)
    is_hypotension         = (t_sbp <= 90).astype(float)
    is_hypertension        = (t_sbp > 220).astype(float)
    is_bradycardia_total   = (t_hr < 40).astype(float)
    is_bradycardia_moderate= ((t_hr >= 40) & (t_hr < 60)).astype(float)
    is_tachycardia_total   = (t_hr > 150).astype(float)
    is_tachycardia_moderate= ((t_hr >= 100) & (t_hr <= 150)).astype(float)
    
    X[:, 15] = is_dyspnea_total
    X[:, 16] = is_dyspnea_moderate
    X[:, 17] = is_bradypnea
    X[:, 18] = is_tachypnea
    X[:, 19] = is_hypotension
    X[:, 20] = is_hypertension
    X[:, 21] = is_bradycardia_total
    X[:, 22] = is_bradycardia_moderate
    X[:, 23] = is_tachycardia_total
    X[:, 24] = is_tachycardia_moderate
    
    # [25..37] Ranges, Distances & Ratios
    X[:, 25] = hr_rng
    X[:, 26] = rr_rng
    X[:, 27] = spo2_rng
    X[:, 28] = sbp_rng
    shock_idx = t_hr / np.where(t_sbp == 0, 1.0, t_sbp)
    X[:, 29] = shock_idx
    X[:, 30] = t_hr - hr_rng
    X[:, 31] = t_sbp - sbp_rng
    X[:, 32] = t_rr - rr_rng
    X[:, 33] = t_o2 - spo2_rng
    X[:, 34] = t_o2 / np.where(t_rr == 0, 1.0, t_rr) # rox_index
    X[:, 35] = spo2_rng / np.where(spo2_max == 0, 1.0, spo2_max) # spo2_drop_ratio
    hr_instab = hr_rng / (t_hr + 1.0)
    X[:, 36] = hr_instab
    X[:, 37] = (t_rr / np.where(t_o2 == 0, 1.0, t_o2)) * 100.0 # bif
    
    # [38..39] Multi-System Derangement Counts (Category 1)
    X[:, 38] = is_dyspnea_total + is_bradypnea + is_tachypnea + is_hypotension + is_bradycardia_total + is_tachycardia_total # n_critical_vitals
    X[:, 39] = (is_dyspnea_total + is_dyspnea_moderate + is_bradypnea + is_tachypnea +
                is_hypotension + is_hypertension + is_bradycardia_total + is_bradycardia_moderate +
                is_tachycardia_total + is_tachycardia_moderate) # n_abnormal_vitals
    
    # [40] Composite Severity Score (NEWS-like, Category 2)
    n_hr  = calc_news_hr(t_hr)
    n_rr  = calc_news_rr(t_rr)
    n_sbp = calc_news_sbp(t_sbp)
    n_o2  = calc_news_spo2(t_o2)
    X[:, 40] = n_hr + n_rr + n_sbp + n_o2 # news_score
    
    # [41] Euclidean Distance from Population Norms (Category 4)
    hr_dev_sq  = ((t_hr - 80.0) / 80.0) ** 2
    sbp_dev_sq = ((t_sbp - 120.0) / 120.0) ** 2
    rr_dev_sq  = ((t_rr - 16.0) / 16.0) ** 2
    o2_dev_sq  = ((t_o2 - 98.0) / 98.0) ** 2
    X[:, 41] = np.sqrt(hr_dev_sq + sbp_dev_sq + rr_dev_sq + o2_dev_sq) # vital_distance
    
    # [42..43] Hemodynamic Stability & Perfusion Gap (Category 6)
    X[:, 42] = (t_hr / 80.0) - (t_sbp / 120.0) # perfusion_gap
    X[:, 43] = shock_idx + hr_instab + (sbp_rng / (t_sbp + 1.0)) # hemo_instability
    
    # [44..46] Respiratory Reserve & Efficiency (Category 7)
    X[:, 44] = np.clip(spo2_min - 90.0, -20.0, 20.0) # resp_reserve
    X[:, 45] = (rr_rng + spo2_rng) * (1.0 + is_tachypnea + is_dyspnea_total) # resp_severity
    X[:, 46] = t_o2 / (t_rr + 1.0) # o2_efficiency
    
    # [47..49] Chief Complaint Interactions (Category 5)
    X[:, 47] = cc_bd * (100.0 - t_o2) # bd_x_o2_deficit
    X[:, 48] = cc_bd * is_tachypnea   # bd_x_tachypnea
    X[:, 49] = cc_bd * (n_rr + n_o2)  # bd_x_news_resp
    
    # [50..52] Age-Vital Sign Interactions (Category 3)
    X[:, 50] = age * (100.0 - t_o2)   # age_o2_interaction
    X[:, 51] = age * t_hr / 100.0     # age_hr_interaction
    X[:, 52] = age * t_sbp / 100.0    # age_sbp_interaction
    
    # [53..56] Individual Squared Deviations (Category 4)
    X[:, 53] = hr_dev_sq
    X[:, 54] = sbp_dev_sq
    X[:, 55] = rr_dev_sq
    X[:, 56] = o2_dev_sq
    
    return X

all_candidate_feature_names = [
    'age', 'cc_breathingdifficulty', 'gender', 'triage_vital_hr', 'triage_vital_sbp', 'triage_vital_rr', 'triage_vital_o2',
    'pulse_min', 'resp_min', 'spo2_min', 'sbp_min', 'pulse_max', 'resp_max', 'spo2_max', 'sbp_max',
    'is_dyspnea_total', 'is_dyspnea_moderate', 'is_bradypnea', 'is_tachypnea', 'is_hypotension', 'is_hypertension',
    'is_bradycardia_total', 'is_bradycardia_moderate', 'is_tachycardia_total', 'is_tachycardia_moderate',
    'hr_range', 'rr_range', 'spo2_range', 'sbp_range',
    'shock_index', 'hr_mid_to_triage', 'sbp_mid_to_triage', 'rr_mid_to_triage', 'spo2_mid_to_triage',
    'rox_index', 'spo2_drop_ratio', 'hr_instability_ratio', 'bif',
    'n_critical_vitals', 'n_abnormal_vitals', 'news_score', 'vital_distance',
    'perfusion_gap', 'hemo_instability', 'resp_reserve', 'resp_severity', 'o2_efficiency',
    'bd_x_o2_deficit', 'bd_x_tachypnea', 'bd_x_news_resp',
    'age_o2_interaction', 'age_hr_interaction', 'age_sbp_interaction',
    'hr_dev_sq', 'sbp_dev_sq', 'rr_dev_sq', 'o2_dev_sq'
]

print(f"Total Candidate Features Initialized: {len(all_candidate_feature_names)} (15 Raw + 42 Engineered)")

X_train_cand_raw = build_candidate_feature_pool(raw_mat_tr)
X_val_cand_raw   = build_candidate_feature_pool(raw_mat_val)
X_test_cand_raw  = build_candidate_feature_pool(raw_mat_ts)

print(f"Feature Matrix Extracted: Train={X_train_cand_raw.shape}, Val={X_val_cand_raw.shape}, Test={X_test_cand_raw.shape}")

In [ ]:
# ---------------------------------------------------------
# Step 2.5: Optuna Feature Selection on Validation Set (No SMOTE, Uncapped Trials)
# ---------------------------------------------------------
def compute_macro_balanced_accuracy(y_true, y_pred):
    classes = [1, 2, 3, 4, 5]
    bal_accs = []
    for cls in classes:
        y_bin_true = (y_true == cls).astype(int)
        y_bin_pred = (y_pred == cls).astype(int)
        tp = np.sum((y_bin_true == 1) & (y_bin_pred == 1))
        fn = np.sum((y_bin_true == 1) & (y_bin_pred == 0))
        fp = np.sum((y_bin_true == 0) & (y_bin_pred == 1))
        tn = np.sum((y_bin_true == 0) & (y_bin_pred == 0))
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        bal_accs.append((rec + spec) / 2.0)
    return np.mean(bal_accs)

# Continuous candidate feature indices for standardization
cont_candidate_indices = set([
    0, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14,
    25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37,
    41, 42, 43, 44, 45, 46, 47, 50, 51, 52, 53, 54, 55, 56
])

# Optuna Objective Function: Evaluated strictly on the Validation Set
def objective(trial):
    # Always include the 15 core raw features
    selected_indices = list(range(15))
    
    # Explore candidate features
    use_vital_flags = trial.suggest_categorical('use_vital_flags', [True, False])
    if use_vital_flags:
        selected_indices.extend(list(range(15, 25)))
        
    use_ranges_ratios = trial.suggest_categorical('use_ranges_ratios', [True, False])
    if use_ranges_ratios:
        selected_indices.extend(list(range(25, 38)))
        
    adv_feature_names = all_candidate_feature_names[38:]
    for offset, feat_name in enumerate(adv_feature_names):
        idx = 38 + offset
        if trial.suggest_categorical(f'use_{feat_name}', [True, False]):
            selected_indices.append(idx)
            
    selected_indices = sorted(list(set(selected_indices)))
    
    # Subset matrices for this trial
    X_tr_sub  = X_train_cand_raw[:, selected_indices].copy()
    X_val_sub = X_val_cand_raw[:, selected_indices].copy()
    
    # Standardize continuous columns in the selected subset
    sub_cont_idx = [i for i, orig_idx in enumerate(selected_indices) if orig_idx in cont_candidate_indices]
    if len(sub_cont_idx) > 0:
        scaler_sub = StandardScaler()
        X_tr_sub[:, sub_cont_idx]  = scaler_sub.fit_transform(X_tr_sub[:, sub_cont_idx])
        X_val_sub[:, sub_cont_idx] = scaler_sub.transform(X_val_sub[:, sub_cont_idx])
        
    # Fast sub-models training WITHOUT SMOTE
    lgb_params_trial = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'learning_rate': 0.05,
        'num_leaves': 31,
        'max_depth': 6,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.8,
        'bagging_freq': 1,
        'verbosity': -1,
        'random_state': 42
    }
    
    # Layer 1: ESI 1 vs (ESI 2..5) - No SMOTE
    m1 = lgb.LGBMClassifier(**lgb_params_trial, n_estimators=60)
    m1.fit(X_tr_sub, (y_train == 1).astype(int), eval_set=[(X_val_sub, (y_val == 1).astype(int))], callbacks=[lgb.early_stopping(8, verbose=False)])
    p1_val = m1.predict_proba(X_val_sub)[:, 1]
    
    # Layer 2: ESI 2,3 vs ESI 4,5 (trained on non-ESI 1) - No SMOTE
    m2_tr  = (y_train != 1); m2_val = (y_val != 1)
    m2 = lgb.LGBMClassifier(**lgb_params_trial, n_estimators=60)
    m2.fit(X_tr_sub[m2_tr], np.isin(y_train[m2_tr], [2, 3]).astype(int), eval_set=[(X_val_sub[m2_val], np.isin(y_val[m2_val], [2, 3]).astype(int))], callbacks=[lgb.early_stopping(8, verbose=False)])
    p2_val = m2.predict_proba(X_val_sub)[:, 1]
    
    # Layer 3A: ESI 2 vs ESI 3 - No SMOTE
    m3a_tr  = np.isin(y_train, [2, 3]); m3a_val = np.isin(y_val, [2, 3])
    m3a = lgb.LGBMClassifier(**lgb_params_trial, n_estimators=60)
    m3a.fit(X_tr_sub[m3a_tr], (y_train[m3a_tr] == 2).astype(int), eval_set=[(X_val_sub[m3a_val], (y_val[m3a_val] == 2).astype(int))], callbacks=[lgb.early_stopping(8, verbose=False)])
    p3a_val = m3a.predict_proba(X_val_sub)[:, 1]
    
    # Layer 3B: ESI 4 vs ESI 5 - No SMOTE
    m3b_tr  = np.isin(y_train, [4, 5]); m3b_val = np.isin(y_val, [4, 5])
    m3b = lgb.LGBMClassifier(**lgb_params_trial, n_estimators=60)
    m3b.fit(X_tr_sub[m3b_tr], (y_train[m3b_tr] == 4).astype(int), eval_set=[(X_val_sub[m3b_val], np.isin(y_val[m3b_val], [4, 5]).astype(int))], callbacks=[lgb.early_stopping(8, verbose=False)])
    p3b_val = m3b.predict_proba(X_val_sub)[:, 1]
    
    # Construct Probability Matrix on Validation Set
    val_probs_t = np.zeros((len(X_val_sub), 5))
    val_probs_t[:, 0] = p1_val
    val_probs_t[:, 1] = (1 - p1_val) * p2_val * p3a_val
    val_probs_t[:, 2] = (1 - p1_val) * p2_val * (1 - p3a_val)
    val_probs_t[:, 3] = (1 - p1_val) * (1 - p2_val) * p3b_val
    val_probs_t[:, 4] = (1 - p1_val) * (1 - p2_val) * (1 - p3b_val)
    
    # Evaluate 5-Fold Cross-Validation on the Validation Set
    skf_opt = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof_preds_val = np.zeros(len(val_probs_t))
    
    for v_tr, v_te in skf_opt.split(val_probs_t, y_val):
        lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
        lr.fit(val_probs_t[v_tr], y_val[v_tr])
        oof_preds_val[v_te] = lr.predict(val_probs_t[v_te])
        
    # Return Macro Balanced Accuracy on the Validation Set
    val_macro_bal_acc = compute_macro_balanced_accuracy(y_val, oof_preds_val)
    return val_macro_bal_acc

N_TRIALS = 50  # Set number of trials to explore combinations fully
print(f"Starting Full Optuna Feature Selection Study ({N_TRIALS} Trials, Uncapped Duration, Evaluated on Validation Set)...")
optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))

# Run till done with timeout=None
study.optimize(objective, n_trials=N_TRIALS, timeout=None, show_progress_bar=True)

print("========================================================================")
print(f"OPTUNA OPTIMIZATION FULLY COMPLETED ({len(study.trials)} Trials Executed)!")
print(f"Best Trial #{study.best_trial.number}: Peak Validation Macro Balanced Accuracy = {study.best_value:.4f}")
print("========================================================================")

In [ ]:
# ---------------------------------------------------------
# Step 2.6: Train Final Production Pipeline with Winning Feature Subset (No SMOTE)
# ---------------------------------------------------------
best_params = study.best_trial.params

selected_indices_best = list(range(15))
if best_params.get('use_vital_flags', True):
    selected_indices_best.extend(list(range(15, 25)))
if best_params.get('use_ranges_ratios', True):
    selected_indices_best.extend(list(range(25, 38)))
    
for offset, feat_name in enumerate(all_candidate_feature_names[38:]):
    idx = 38 + offset
    if best_params.get(f'use_{feat_name}', False):
        selected_indices_best.append(idx)
        
selected_indices_best = sorted(list(set(selected_indices_best)))
final_feature_names   = [all_candidate_feature_names[i] for i in selected_indices_best]

print(f"Total Optimal Features Selected: {len(final_feature_names)} / {len(all_candidate_feature_names)}")
print("Selected Winning Features:")
for i, fn in enumerate(final_feature_names, 1):
    print(f"  {i:2d}. {fn}")

# Extract final subsets
X_train_best = X_train_cand_raw[:, selected_indices_best].copy()
X_val_best   = X_val_cand_raw[:, selected_indices_best].copy()
X_test_best  = X_test_cand_raw[:, selected_indices_best].copy()

cont_cols_best = [i for i, orig_idx in enumerate(selected_indices_best) if orig_idx in cont_candidate_indices]

scaler_final = StandardScaler()
if len(cont_cols_best) > 0:
    X_train_best[:, cont_cols_best] = scaler_final.fit_transform(X_train_best[:, cont_cols_best])
    X_val_best[:, cont_cols_best]   = scaler_final.transform(X_val_best[:, cont_cols_best])
    X_test_best[:, cont_cols_best]  = scaler_final.transform(X_test_best[:, cont_cols_best])

# Train Full Production Sub-Models Directly (No SMOTE)
lgb_params_final = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': 6,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'verbosity': -1,
    'random_state': 42
}

print("\nTraining Production LightGBM Sub-Models on Optimal Feature Subset (No SMOTE)...")
l1_prod = lgb.LGBMClassifier(**lgb_params_final, n_estimators=100)
l1_prod.fit(X_train_best, (y_train == 1).astype(int), eval_set=[(X_val_best, (y_val == 1).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

m2_tr_f  = (y_train != 1); m2_val_f = (y_val != 1)
l2_prod = lgb.LGBMClassifier(**lgb_params_final, n_estimators=100)
l2_prod.fit(X_train_best[m2_tr_f], np.isin(y_train[m2_tr_f], [2, 3]).astype(int), eval_set=[(X_val_best[m2_val_f], np.isin(y_val[m2_val_f], [2, 3]).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

m3a_tr_f  = np.isin(y_train, [2, 3]); m3a_val_f = np.isin(y_val, [2, 3])
l3a_prod = lgb.LGBMClassifier(**lgb_params_final, n_estimators=100)
l3a_prod.fit(X_train_best[m3a_tr_f], (y_train[m3a_tr_f] == 2).astype(int), eval_set=[(X_val_best[m3a_val_f], (y_val[m3a_val_f] == 2).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

m3b_tr_f  = np.isin(y_train, [4, 5]); m3b_val_f = np.isin(y_val, [4, 5])
l3b_prod = lgb.LGBMClassifier(**lgb_params_final, n_estimators=100)
l3b_prod.fit(X_train_best[m3b_tr_f], (y_train[m3b_tr_f] == 4).astype(int), eval_set=[(X_val_best[m3b_val_f], np.isin(y_val[m3b_val_f], [4, 5]).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

# Validation Set Probabilities & Final Meta-Learner
p1_v = l1_prod.predict_proba(X_val_best)[:, 1]
p2_v = l2_prod.predict_proba(X_val_best)[:, 1]
p3a_v = l3a_prod.predict_proba(X_val_best)[:, 1]
p3b_v = l3b_prod.predict_proba(X_val_best)[:, 1]

val_probs_final = np.zeros((len(X_val_best), 5))
val_probs_final[:, 0] = p1_v
val_probs_final[:, 1] = (1 - p1_v) * p2_v * p3a_v
val_probs_final[:, 2] = (1 - p1_v) * p2_v * (1 - p3a_v)
val_probs_final[:, 3] = (1 - p1_v) * (1 - p2_v) * p3b_val if 'p3b_val' in locals() else (1 - p1_v) * (1 - p2_v) * p3b_v
val_probs_final[:, 4] = (1 - p1_v) * (1 - p2_v) * (1 - p3b_v)

meta_logreg_exp = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
meta_logreg_exp.fit(val_probs_final, y_val)

# Holdout Test Predictions
p1_ts = l1_prod.predict_proba(X_test_best)[:, 1]
p2_ts = l2_prod.predict_proba(X_test_best)[:, 1]
p3a_ts = l3a_prod.predict_proba(X_test_best)[:, 1]
p3b_ts = l3b_prod.predict_proba(X_test_best)[:, 1]

test_probs_prod_exp = np.zeros((len(X_test_best), 5))
test_probs_prod_exp[:, 0] = p1_ts
test_probs_prod_exp[:, 1] = (1 - p1_ts) * p2_ts * p3a_ts
test_probs_prod_exp[:, 2] = (1 - p1_ts) * p2_ts * (1 - p3a_ts)
test_probs_prod_exp[:, 3] = (1 - p1_ts) * (1 - p2_ts) * p3b_ts
test_probs_prod_exp[:, 4] = (1 - p1_ts) * (1 - p2_ts) * (1 - p3b_ts)

# Export Bundle
deploy_dir = '../deploy' if os.path.exists('../deploy') else 'deploy'
os.makedirs(deploy_dir, exist_ok=True)

bundle_data_exp = {
    'l1_prod': l1_prod,
    'l2_prod': l2_prod,
    'l3a_prod': l3a_prod,
    'l3b_prod': l3b_prod,
    'meta_logreg': meta_logreg_exp,
    'scaler_means': scaler_final.mean_ if len(cont_cols_best) > 0 else None,
    'scaler_sds': scaler_final.scale_ if len(cont_cols_best) > 0 else None,
    'cont_cols_idx': cont_cols_best,
    'feature_names': final_feature_names,
    'selected_indices': selected_indices_best
}

with open(os.path.join(deploy_dir, 'py_oof_stacking_bundle_exp.pkl'), 'wb') as f:
    pickle.dump(bundle_data_exp, f)

print(f"Optimized Model Bundle saved to: {os.path.join(deploy_dir, 'py_oof_stacking_bundle_exp.pkl')}")

In [ ]:
# ---------------------------------------------------------
# Step 3: Holdout Test Set Evaluation & Detailed Per-Class Breakdown
# ---------------------------------------------------------
preds_meta_exp = meta_logreg_exp.predict(test_probs_prod_exp)
probs_meta_exp = meta_logreg_exp.predict_proba(test_probs_prod_exp)

def get_per_class_breakdown(y_true, y_pred, probs, pipeline_name):
    classes = [1, 2, 3, 4, 5]
    rows = []
    recalls, specs, bal_accs, aucs = [], [], [], []
    for idx, cls in enumerate(classes):
        y_bin_true = (y_true == cls).astype(int)
        y_bin_pred = (y_pred == cls).astype(int)
        tp = np.sum((y_bin_true == 1) & (y_bin_pred == 1))
        fn = np.sum((y_bin_true == 1) & (y_bin_pred == 0))
        fp = np.sum((y_bin_true == 0) & (y_bin_pred == 1))
        tn = np.sum((y_bin_true == 0) & (y_bin_pred == 0))
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        bal  = (rec + spec) / 2.0
        try: auc = roc_auc_score(y_bin_true, probs[:, idx])
        except Exception: auc = 0.0
        recalls.append(rec); specs.append(spec); bal_accs.append(bal); aucs.append(auc)
        rows.append({
            'Pipeline': pipeline_name,
            'Class': f'ESI_{cls}',
            'Recall': round(rec, 4),
            'Specificity': round(spec, 4),
            'Balanced_Accuracy': round(bal, 4),
            'ROC_AUC': round(auc, 4)
        })
    rows.append({
        'Pipeline': pipeline_name,
        'Class': 'Macro_Average',
        'Recall': round(np.mean(recalls), 4),
        'Specificity': round(np.mean(specs), 4),
        'Balanced_Accuracy': round(np.mean(bal_accs), 4),
        'ROC_AUC': round(np.mean(aucs), 4)
    })
    return pd.DataFrame(rows)

report_df_exp = get_per_class_breakdown(y_test, preds_meta_exp, probs_meta_exp, 'Optuna_Optimized_Stacking_Pipeline')

print("========================================================================================")
print("   HOLDOUT TEST SET REPORT: OPTUNA OPTIMIZED STACKING PIPELINE (BALANCED ACCURACY)")
print("========================================================================================")
print(report_df_exp.to_string(index=False))
print("========================================================================================\n")

reports_dir = '../reports' if os.path.exists('../reports') else 'reports'
os.makedirs(reports_dir, exist_ok=True)

report_df_exp.to_csv(os.path.join(reports_dir, 'oof_multinomial_logistic_stacking_report_exp.csv'), index=False)
print(f"Report saved to {os.path.join(reports_dir, 'oof_multinomial_logistic_stacking_report_exp.csv')}")

In [ ]:
# ---------------------------------------------------------
# Step 4: Confusion Matrix Graph for Experimental Benchmark
# ---------------------------------------------------------
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

plots_dir = '../plots' if os.path.exists('../plots') else 'plots'
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(os.path.join(plots_dir, 'image'), exist_ok=True)

esi_labels = [f"ESI {i}" for i in range(1, 6)]

cm_meta      = confusion_matrix(y_test, preds_meta_exp, labels=[1, 2, 3, 4, 5])
cm_meta_norm = cm_meta.astype('float') / cm_meta.sum(axis=1)[:, np.newaxis]

fig, ax = plt.subplots(figsize=(9, 7))

annot_meta = np.empty_like(cm_meta, dtype=object)
for i in range(5):
    for j in range(5):
        annot_meta[i, j] = f"{cm_meta[i, j]}\n({cm_meta_norm[i, j]*100:.1f}%)"

sns.heatmap(cm_meta_norm, annot=annot_meta, fmt='', cmap='Greens', cbar=True,
            xticklabels=esi_labels, yticklabels=esi_labels, ax=ax, vmin=0, vmax=1)
ax.set_title('Optuna Optimized Stacking Pipeline\nHoldout Test Confusion Matrix', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Predicted ESI Level', fontsize=11, fontweight='bold')
ax.set_ylabel('True ESI Level', fontsize=11, fontweight='bold')

plt.tight_layout()

cm_path = os.path.join(plots_dir, 'holdout_test_confusion_matrix_exp.png')
plt.savefig(cm_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'holdout_test_confusion_matrix_exp.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"Confusion Matrix Graph saved to: {cm_path}")

In [ ]:
# ---------------------------------------------------------
# Step 5: Density Data Distribution Graphs for Selected Features
# ---------------------------------------------------------
import os
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

plots_dir = '../plots' if os.path.exists('../plots') else 'plots'
density_dir = os.path.join(plots_dir, 'density_exp')
density_img_dir = os.path.join(plots_dir, 'image', 'density_exp')
os.makedirs(density_dir, exist_ok=True)
os.makedirs(density_img_dir, exist_ok=True)

df_raw_best = pd.DataFrame(X_train_best, columns=final_feature_names)
df_raw_best['ESI'] = [f"ESI {k}" for k in y_train]

esi_palette = {
    'ESI 1': '#d62728',  # Red (Resuscitation)
    'ESI 2': '#ff7f0e',  # Orange (Emergent)
    'ESI 3': '#2ca02c',  # Green (Urgent)
    'ESI 4': '#1f77b4',  # Blue (Less Urgent)
    'ESI 5': '#9467bd'   # Purple (Non-urgent)
}

print(f"Saving density distribution plots for {len(final_feature_names)} selected features to: {density_dir}")

for feat in final_feature_names:
    fig, ax = plt.subplots(figsize=(8, 5))
    
    # Categorical/Binary features
    if feat in ['gender', 'cc_breathingdifficulty'] or feat.startswith('is_') or feat.startswith('bd_x_tachypnea'):
        prop_df = df_raw_best.groupby('ESI')[feat].mean().reset_index(name='Proportion')
        sns.barplot(data=prop_df, x='ESI', y='Proportion', palette=esi_palette, ax=ax, edgecolor='black')
        ax.set_title(f"{feat} (Prevalence by ESI Level)", fontsize=13, fontweight='bold', pad=12)
        ax.set_xlabel("ESI Level", fontsize=11, fontweight='bold')
        ax.set_ylabel("Prevalence / Proportion", fontsize=11, fontweight='bold')
        for p in ax.patches:
            ax.annotate(f"{p.get_height()*100:.1f}%",
                        (p.get_x() + p.get_width() / 2., p.get_height()),
                        ha='center', va='bottom', fontsize=10, fontweight='bold', xytext=(0, 2),
                        textcoords='offset points')
    else:
        sns.kdeplot(
            data=df_raw_best,
            x=feat,
            hue='ESI',
            palette=esi_palette,
            common_norm=False,
            fill=True,
            alpha=0.20,
            linewidth=2.0,
            ax=ax
        )
        ax.set_title(f"Feature Density Distribution: {feat}", fontsize=13, fontweight='bold', pad=12)
        ax.set_xlabel(feat, fontsize=11, fontweight='bold')
        ax.set_ylabel("Density", fontsize=11, fontweight='bold')
    
    ax.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    
    out_file = f"density_{feat}.png"
    plt.savefig(os.path.join(density_dir, out_file), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(density_img_dir, out_file), dpi=300, bbox_inches='tight')
    plt.close()

print(f"All {len(final_feature_names)} density plots successfully saved.")

In [ ]:
# ---------------------------------------------------------
# Step 6: Multiclass ROC-AUC Curve Analysis (Holdout Test Benchmark)
# ---------------------------------------------------------
import os
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

plots_dir = '../plots' if os.path.exists('../plots') else 'plots'
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(os.path.join(plots_dir, 'image'), exist_ok=True)

classes = [1, 2, 3, 4, 5]
y_test_bin = label_binarize(y_test, classes=classes)
n_classes  = len(classes)

fpr = dict()
tpr = dict()
roc_auc = dict()

for i, cls in enumerate(classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], probs_meta_exp[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

fpr["micro"], tpr["micro"], _ = roc_curve(y_test_bin.ravel(), probs_meta_exp.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(n_classes):
    mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
mean_tpr /= n_classes

fpr["macro"] = all_fpr
tpr["macro"] = mean_tpr
roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

plt.figure(figsize=(9, 8))

esi_colors = {
    0: '#d62728',  # ESI 1: Red
    1: '#ff7f0e',  # ESI 2: Orange
    2: '#2ca02c',  # ESI 3: Green
    3: '#1f77b4',  # ESI 4: Blue
    4: '#9467bd'   # ESI 5: Purple
}

plt.plot(fpr["micro"], tpr["micro"],
         label=f"Micro-Average (AUC = {roc_auc['micro']:.4f})",
         color='#e377c2', linestyle=':', linewidth=2.5)

plt.plot(fpr["macro"], tpr["macro"],
         label=f"Macro-Average (AUC = {roc_auc['macro']:.4f})",
         color='#17becf', linestyle='--', linewidth=2.5)

for i, cls in enumerate(classes):
    plt.plot(fpr[i], tpr[i], color=esi_colors[i], linewidth=2.0,
             label=f"ESI {cls} (AUC = {roc_auc[i]:.4f})")

plt.plot([0, 1], [0, 1], 'k--', color='gray', linewidth=1.2, label='Random Guess (AUC = 0.5000)')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12, fontweight='bold')
plt.ylabel('True Positive Rate (Recall / Sensitivity)', fontsize=12, fontweight='bold')
plt.title('Optuna Stacking Pipeline\nHoldout Test ROC-AUC Curves (One-vs-Rest)', fontsize=14, fontweight='bold', pad=12)
plt.legend(loc="lower right", fontsize=10.5, frameon=True, framealpha=0.95)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
roc_plot_path = os.path.join(plots_dir, 'holdout_test_roc_auc_curve_exp.png')
plt.savefig(roc_plot_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'holdout_test_roc_auc_curve_exp.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"ROC-AUC Curve Graph saved to: {roc_plot_path}")

In [ ]:
# ---------------------------------------------------------
# Step 7: Optuna Optimization Trajectory & Feature Importance Graph
# ---------------------------------------------------------
import matplotlib.pyplot as plt
import seaborn as sns

trial_numbers = [t.number for t in study.trials if t.value is not None]
trial_values  = [t.value for t in study.trials if t.value is not None]
running_max   = np.maximum.accumulate(trial_values)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# 1. Optimization Trajectory
ax1.scatter(trial_numbers, trial_values, color='#1f77b4', alpha=0.7, label='Trial Macro Balanced Acc')
ax1.plot(trial_numbers, running_max, color='#d62728', linewidth=2.2, label='Best Macro Balanced Acc')
ax1.set_title('Optuna Feature Selection Optimization History', fontsize=13, fontweight='bold', pad=12)
ax1.set_xlabel('Trial Number', fontsize=11, fontweight='bold')
ax1.set_ylabel('Validation Macro Balanced Accuracy', fontsize=11, fontweight='bold')
ax1.legend(loc='lower right', frameon=True)
ax1.grid(True, linestyle='--', alpha=0.4)

# 2. Top Selected Recommended Features in Winning Trials
feat_counts = {}
for t in study.trials:
    if t.value is not None and t.value >= np.median(trial_values):
        for param_k, param_v in t.params.items():
            if param_k.startswith('use_') and param_v is True:
                f_label = param_k.replace('use_', '')
                feat_counts[f_label] = feat_counts.get(f_label, 0) + 1

if len(feat_counts) > 0:
    sorted_feats = sorted(feat_counts.items(), key=lambda x: x[1], reverse=True)[:15]
    labels = [k[0] for k in sorted_feats]
    counts = [k[1] for k in sorted_feats]
    
    sns.barplot(x=counts, y=labels, palette='viridis', ax=ax2, edgecolor='black')
    ax2.set_title('Top Candidate Features Selected in High-Performing Trials', fontsize=13, fontweight='bold', pad=12)
    ax2.set_xlabel('Selection Frequency (Above-Median Trials)', fontsize=11, fontweight='bold')
    ax2.set_ylabel('Feature Name', fontsize=11, fontweight='bold')
    ax2.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
optuna_plot_path = os.path.join(plots_dir, 'optuna_feature_optimization_history.png')
plt.savefig(optuna_plot_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'optuna_feature_optimization_history.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"Optuna Optimization Trajectory Graph saved to: {optuna_plot_path}")